<a href="https://colab.research.google.com/github/bnsreenu/python_for_image_processing_APEER/blob/master/365_Correlation_Analysis_Part_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


https://youtu.be/XGjOuVKqK9I

# Correlation Analysis: Understanding Relationships in Data - Part 1
**Part 7 of the Statistical Analysis in Python Tutorial Series**

## Overview

Correlation analysis is one of the most fundamental and widely-used statistical techniques for exploring relationships between variables. This comprehensive tutorial takes you from basic concepts to advanced applications using real-world penguin morphology data.

---


## Dataset: Palmer Penguins

We'll use the famous Palmer Penguins dataset, which provides:
- **344 penguins** across 3 species (Adelie, Chinstrap, Gentoo)
- **Morphological measurements**: bill length/depth, flipper length, body mass
- **Real biological relationships** to explore and interpret
- **Natural confounding factors** (species effects) to understand


---

## Tutorial Structure

**Part 1: Foundation**
- Data exploration and quality assessment
- Visual demonstration of correlation concepts
- Basic Pearson correlation calculations
- Systematic testing of correlation assumptions
- Diagnostic plotting and interpretation
- Spearman correlation as robust alternative

**Part 2: Partial Correlations**  
- Partial correlations controlling for confounding
- Correlation matrices and multiple relationships
- Visualizing correlations

**Part 3: Advanced Correlation Analysis**
- Confidence Intervals
- Statistical Significance

**Part 4: Correlation vs. causation**
- Correlation vs. causation analysis
- Spurious correlation examples

---



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, kendalltau
import statsmodels.api as sm
from statsmodels.stats.correlation_tools import corr_nearest, corr_clipped
import warnings
from itertools import combinations
import requests
from io import StringIO

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LinearRegression


from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from statsmodels.stats.multitest import multipletests
from sklearn.utils import resample
import math




# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set visualization style consistent with previous tutorials
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

In [ ]:
# Load palmer penguin dataset from online source
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"

penguins = pd.read_csv(url)


# Display basic information about the dataset
print("\nDataset Information:")
print("=" * 25)
print(f"Shape: {penguins.shape}")
print(f"Columns: {list(penguins.columns)}")

In [ ]:
# Check for missing values
print("\nMissing values per column:")
missing_counts = penguins.isnull().sum()
for col, count in missing_counts.items():
    print(f"  {col}: {count}")

# Remove rows with missing values for correlation analysis
penguins_clean = penguins.dropna()
print(f"\nAfter removing missing values: {penguins_clean.shape[0]} complete observations")

In [ ]:
# Define continuous variables for correlation analysis
continuous_vars = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
categorical_vars = ['species', 'island', 'sex']



# 1. Introduction to Correlation Analysis

## What is Correlation?

Correlation measures the strength and direction of a linear relationship between two continuous variables.
It helps us understand how variables move together:

- **Positive correlation**: As one variable increases, the other tends to increase
- **Negative correlation**: As one variable increases, the other tends to decrease
- **Zero correlation**: No linear relationship between variables

## Key Properties of Correlation:

1. **Range**: Correlation coefficients typically range from -1 to +1
2. **Magnitude**: |r| indicates strength of relationship
3. **Sign**: + or - indicates direction of relationship
4. **No causation**: Correlation does not imply causation

## Interpretation Guidelines:
- |r| ≥ 0.7: Strong relationship
- 0.3 ≤ |r| < 0.7: Moderate relationship
- |r| < 0.3: Weak relationship

Let's start by exploring our penguin data to understand the relationships between physical measurements.


# Exploratory Data Analysis for Correlation Studies

## Why EDA Before Correlation Analysis?

Understanding data characteristics determines which correlation methods to use and how to interpret results.

---

## What This Analysis Checks

### 1. **Normality Assessment**
- **Normal data**: Use Pearson correlation
- **Non-normal data**: Use Spearman correlation
- **Tools**: Shapiro-Wilk test, Q-Q plots, histograms

### 2. **Outlier Detection**
- **Impact**: Outliers can distort correlation coefficients
- **Methods**: Box plots, IQR method
- **Decision**: Remove, transform, or use robust methods

### 3. **Species Differences**
- **Large differences**: Species likely confounds correlations
- **Small differences**: Correlations more likely genuine
- **Implication**: Determines need for partial correlation analysis

### 4. **Data Quality**
- **Coefficient of variation**: Measurement reliability
- **Distribution patterns**: Data transformation needs

---

## Key Decisions This EDA Will Guide

**Correlation Method**:
- Normal data → Pearson correlation
- Non-normal data → Spearman correlation

**Analysis Approach**:
- Large species effects → Use partial correlations
- Few outliers → Standard analysis
- Many outliers → Robust methods

This foundation ensures our correlation analysis uses appropriate methods and produces reliable results.

In [ ]:
def explore_correlation_data(data, continuous_vars):
    """
    Exploration of the dataset focusing on relationships between continuous variables.

    """
    print("\n" + "=" * 70)
    print("EXPLORATORY DATA ANALYSIS FOR CORRELATION")
    print("=" * 70)

    # Basic descriptive statistics
    print("\nDescriptive Statistics for Continuous Variables:")
    print("-" * 50)

    desc_stats = data[continuous_vars].describe().round(2)
    print(desc_stats)

    # Calculate coefficient of variation for each variable
    # Remember that CV helps to understand the extent of dispersion or variability around the mean of a dataset.
    print(f"\nCoefficient of Variation (CV = std/mean):")
    print("-" * 40)
    for var in continuous_vars:
        mean_val = data[var].mean()
        std_val = data[var].std()
        cv = (std_val / mean_val) * 100
        print(f"{var:<20}: {cv:>6.1f}%")

    # Check distributions for normality (important for Pearson correlation)
    print(f"\nNormality Assessment (Shapiro-Wilk Test):")
    print("-" * 45)
    normality_results = {}

    for var in continuous_vars:
        # Use sample if data is too large for Shapiro-Wilk
        sample_data = data[var].dropna()
        if len(sample_data) > 5000:
            sample_data = sample_data.sample(5000, random_state=42)

        stat, p_value = stats.shapiro(sample_data)
        normality_results[var] = {'statistic': stat, 'p_value': p_value}

        interpretation = "Normal" if p_value > 0.05 else "Non-normal"
        print(f"{var:<20}: W = {stat:.4f}, p = {p_value:.4f} ({interpretation})")

    # Create different visualizations
    n_vars = len(continuous_vars)
    fig, axes = plt.subplots(3, n_vars, figsize=(4*n_vars, 12))
    fig.suptitle('Exploratory Data Analysis: Continuous Variables', fontsize=16)

    for i, var in enumerate(continuous_vars):
        var_data = data[var].dropna()

        # Row 1: Histograms with normal overlay
        axes[0, i].hist(var_data, bins=30, density=True, alpha=0.7,
                       color='skyblue', edgecolor='black')

        # Overlay normal distribution
        mu, sigma = stats.norm.fit(var_data)
        x = np.linspace(var_data.min(), var_data.max(), 100)
        axes[0, i].plot(x, stats.norm.pdf(x, mu, sigma), 'r-',
                       linewidth=2, label='Normal fit')

        axes[0, i].set_title(f'Distribution: {var}')
        axes[0, i].set_xlabel(var)
        axes[0, i].set_ylabel('Density')
        axes[0, i].legend()
        axes[0, i].grid(True, alpha=0.3)

        # Row 2: Q-Q plots for normality
        stats.probplot(var_data, dist="norm", plot=axes[1, i])
        axes[1, i].set_title(f'Q-Q Plot: {var}')
        axes[1, i].grid(True, alpha=0.3)

        # Row 3: Box plots
        axes[2, i].boxplot(var_data, patch_artist=True,
                          boxprops=dict(facecolor='lightblue', alpha=0.7))
        axes[2, i].set_title(f'Box Plot: {var}')
        axes[2, i].set_ylabel(var)
        axes[2, i].grid(True, alpha=0.3)

        # Add outlier information
        Q1 = var_data.quantile(0.25)
        Q3 = var_data.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = var_data[(var_data < lower_bound) | (var_data > upper_bound)]

        axes[2, i].text(0.5, 0.95, f'Outliers: {len(outliers)}',
                       transform=axes[2, i].transAxes, ha='center', va='top',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.show()

    # Identify potential outliers
    print(f"\nOutlier Analysis:")
    print("-" * 20)

    outlier_summary = {}
    for var in continuous_vars:
        var_data = data[var].dropna()
        Q1 = var_data.quantile(0.25)
        Q3 = var_data.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = var_data[(var_data < lower_bound) | (var_data > upper_bound)]
        outlier_percentage = (len(outliers) / len(var_data)) * 100

        outlier_summary[var] = {
            'count': len(outliers),
            'percentage': outlier_percentage,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound
        }

        print(f"{var:<20}: {len(outliers):>3} outliers ({outlier_percentage:>5.1f}%)")

    # Species-specific analysis
    if 'species' in data.columns:
        print(f"\nSpecies-specific Means:")
        print("-" * 25)
        species_means = data.groupby('species')[continuous_vars].mean().round(2)
        print(species_means)

        # Visualize species differences
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('Variable Distributions by Species', fontsize=16)

        for i, var in enumerate(continuous_vars):
            row = i // 2
            col = i % 2

            sns.boxplot(data=data, x='species', y=var, ax=axes[row, col])
            axes[row, col].set_title(f'{var} by Species')
            axes[row, col].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()

    return {
        'descriptive_stats': desc_stats,
        'normality_results': normality_results,
        'outlier_summary': outlier_summary,
        'species_means': species_means if 'species' in data.columns else None
    }

# Step 1: Explore the data
exploration_results = explore_correlation_data(penguins_clean, continuous_vars)

# Correlation Concepts: Visual Demonstration
This section builds intuition about correlation by showing what different correlation strengths actually look like in scatter plots. We'll generate simulated data with known correlations (ranging from r = -0.9 to r = +0.9) so you can see how the strength and direction of relationships appear visually. This visual foundation will help you better interpret the correlations we find in the real penguin data that follows.

**The correlated data is generated by:**

Constructing a covariance matrix where the off-diagonal value is the target correlation r

Sampling from a multivariate normal distribution using that covariance structure. The multivariate normal distribution inherently supports specifying correlation (or covariance) between variables.



This approach guarantees that the resulting synthetic data pairs (x, y) have the desired correlation structure.

In [ ]:
def demonstrate_correlation_concepts():
    """
    Demonstrate correlation concepts with simulated examples to build intuition.
    We will then look at our penguin dataset
    """
    print("\n" + "=" * 70)
    print("CORRELATION CONCEPTS: VISUAL DEMONSTRATION")
    print("=" * 70)

    # Generate example data with different correlation strengths
    np.random.seed(42)
    n = 100
    x = np.random.normal(0, 1, n)

    # Create different correlation scenarios
    correlations = [0.9, 0.5, 0.0, -0.5, -0.9]
    correlation_data = {}

    for r in correlations:
        # Generate correlated data using Cholesky decomposition
        cov_matrix = np.array([[1, r], [r, 1]])
        data_pair = np.random.multivariate_normal([0, 0], cov_matrix, n)
        correlation_data[r] = {'x': data_pair[:, 0], 'y': data_pair[:, 1]}

    # Create visualization
    fig, axes = plt.subplots(1, len(correlations), figsize=(20, 4))
    fig.suptitle('Correlation Strength Examples', fontsize=16)

    for i, r in enumerate(correlations):
        x_data = correlation_data[r]['x']
        y_data = correlation_data[r]['y']

        # Calculate actual correlation
        actual_r, _ = pearsonr(x_data, y_data)

        # Create scatter plot
        axes[i].scatter(x_data, y_data, alpha=0.6, s=30)

        # Add regression line
        slope, intercept, _, _, _ = stats.linregress(x_data, y_data)
        line_x = np.linspace(x_data.min(), x_data.max(), 100)
        line_y = slope * line_x + intercept
        axes[i].plot(line_x, line_y, 'r-', linewidth=2)

        # Formatting
        axes[i].set_title(f'Target r = {r}\nActual r = {actual_r:.3f}')
        axes[i].set_xlabel('Variable X')
        if i == 0:
            axes[i].set_ylabel('Variable Y')
        axes[i].grid(True, alpha=0.3)
        axes[i].set_xlim(-3, 3)
        axes[i].set_ylim(-3, 3)

    plt.tight_layout()
    plt.show()

    # Interpretation guide
    print(f"\nCorrelation Interpretation Guide:")
    print("-" * 35)
    interpretation_guide = {
        "Perfect positive (r = +1.0)": "Y increases exactly as X increases",
        "Strong positive (r = +0.7 to +0.9)": "Y tends to increase as X increases",
        "Moderate positive (r = +0.3 to +0.7)": "Y somewhat increases as X increases",
        "Weak (r = -0.3 to +0.3)": "Little to no linear relationship",
        "Moderate negative (r = -0.7 to -0.3)": "Y somewhat decreases as X increases",
        "Strong negative (r = -0.9 to -0.7)": "Y tends to decrease as X increases",
        "Perfect negative (r = -1.0)": "Y decreases exactly as X increases"
    }

    for strength, description in interpretation_guide.items():
        print(f"{strength:<30}: {description}")


  # Step 2: Demonstrate correlation concepts
demonstrate_correlation_concepts()

# Basic Pearson Correlations Analysis

## What We're Calculating

This section computes **Pearson correlations between all pairs** of continuous variables in our penguin dataset and provides a comprehensive interpretation of the relationships.

---

## Key Outputs

### 1. **Correlation Matrix**
- Shows all pairwise correlations in a structured table
- Values range from -1 to +1
- Diagonal is always 1.0 (variables correlate perfectly with themselves)

### 2. **Statistical Significance Testing**
- **p-values**: Test whether correlations are significantly different from zero
- **Sample sizes**: Number of complete observations for each pair
- **Significance threshold**: p < 0.05 indicates non-random relationship

### 3. **Strength Classification**
- **Strong**: |r| ≥ 0.7 (variables move together closely)
- **Moderate**: 0.3 ≤ |r| < 0.7 (noticeable relationship)
- **Weak**: |r| < 0.3 (little linear relationship)

### 4. **Visual Heatmap**
- Color-coded correlation matrix
- **Red**: Negative correlations
- **Blue**: Positive correlations  
- **Intensity**: Stronger correlations = darker colors

---

## What to Look For

**Strong Correlations**: Variables that might be measuring similar biological traits or have functional relationships

**Unexpected Patterns**: Correlations that don't make biological sense (potential confounding)

**Significance vs. Strength**: Large correlations that aren't significant (small sample) or small correlations that are significant (large sample)

This foundation analysis will guide our deeper investigation into species effects and causal relationships.

## Additional explanation, if you are still confused...

**What the correlation is doing:**

- Takes ALL penguins (regardless of species - Adelie, Chinstrap, Gentoo)
- For each individual penguin, looks at both bill_length and bill_depth (or any two variables)
- Asks the question: "Across all these penguins, when bill_length is higher, does bill_depth tend to be higher too?"
- Quantifies this relationship using Pearson's correlation formula
- The correlation pools all species together, so it's asking: "In general, across all penguin types, do longer bills tend to be deeper?"

But this can sometimes be misleading because:

Different species might have systematically different bill shapes. The correlation might be driven by between-species differences rather than within-species relationships

That's why we will later look at Partial correlations (controlling for species)

In [ ]:
def calculate_basic_correlations(data, continuous_vars):
    """
    Calculate and interpret basic Pearson correlations between continuous variables.

    """
    print("\n" + "=" * 70)
    print("BASIC PEARSON CORRELATIONS")
    print("=" * 70)

    # Calculate correlation matrix
    corr_matrix = data[continuous_vars].corr()

    print("Correlation Matrix:")
    print("-" * 20)
    print(corr_matrix.round(3))

    # Extract unique pairs and their correlations
    print(f"\nPairwise Correlations:")
    print("-" * 25)

    correlation_results = []

    for i in range(len(continuous_vars)):
        for j in range(i+1, len(continuous_vars)):
            var1 = continuous_vars[i]
            var2 = continuous_vars[j]

            # FIXED: Calculate correlation with proper alignment
            # Get valid pairs first to ensure alignment
            valid_pairs = data[[var1, var2]].dropna()

            # Calculate correlation and significance test on aligned data
            r, p_value = pearsonr(valid_pairs[var1], valid_pairs[var2])

            # Calculate sample size
            n = len(valid_pairs)

            # Interpret strength
            if abs(r) >= 0.7:
                strength = "Strong"
            elif abs(r) >= 0.3:
                strength = "Moderate"
            else:
                strength = "Weak"

            # Interpret direction
            direction = "Positive" if r > 0 else "Negative"

            # Significance
            significance = "Significant" if p_value < 0.05 else "Not significant"

            correlation_results.append({
                'var1': var1,
                'var2': var2,
                'correlation': r,
                'p_value': p_value,
                'n': n,
                'strength': strength,
                'direction': direction,
                'significance': significance
            })

            print(f"{var1} ↔ {var2}:")
            print(f"  r = {r:>7.3f}, p = {p_value:>7.4f}, n = {n:>3}")
            print(f"  {strength} {direction.lower()} correlation ({significance.lower()})")
            print()

    # Create correlation heatmap
    plt.figure(figsize=(10, 8))

    # Create mask for upper triangle to show only lower triangle
    #mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

    # Create heatmap
    sns.heatmap(corr_matrix,
                #mask=mask,
                annot=True,
                cmap='RdBu_r',
                center=0,
                square=True,
                fmt='.3f',
                cbar_kws={'label': 'Pearson Correlation Coefficient'})

    plt.title('Correlation Matrix Heatmap', fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()

    # Summary statistics
    print(f"Correlation Summary:")
    print("-" * 20)

    correlations_flat = [abs(result['correlation']) for result in correlation_results]
    print(f"Number of correlations: {len(correlations_flat)}")
    print(f"Mean |correlation|: {np.mean(correlations_flat):.3f}")
    print(f"Max |correlation|: {np.max(correlations_flat):.3f}")
    print(f"Min |correlation|: {np.min(correlations_flat):.3f}")

    # Count by strength
    strength_counts = {}
    for result in correlation_results:
        strength = result['strength']
        strength_counts[strength] = strength_counts.get(strength, 0) + 1

    print(f"\nCorrelations by strength:")
    for strength, count in strength_counts.items():
        percentage = (count / len(correlation_results)) * 100
        print(f"  {strength}: {count} ({percentage:.1f}%)")

    # Significant correlations
    significant_corrs = [r for r in correlation_results if r['significance'] == 'Significant']
    print(f"\nSignificant correlations: {len(significant_corrs)} out of {len(correlation_results)}")

    return {
        'correlation_matrix': corr_matrix,
        'pairwise_results': correlation_results,
        'summary_stats': {
            'mean_abs_correlation': np.mean(correlations_flat),
            'max_abs_correlation': np.max(correlations_flat),
            'min_abs_correlation': np.min(correlations_flat),
            'strength_distribution': strength_counts,
            'significant_count': len(significant_corrs)
        }
    }


# Step 3: Calculate basic correlations
correlation_results = calculate_basic_correlations(penguins_clean, continuous_vars)

# Assumption Checking for Correlation Analysis

## Why Check Assumptions?

**Pearson correlation makes specific assumptions about the data.** When these assumptions are violated, the correlation coefficient can be misleading or unreliable. This section systematically tests each assumption and provides guidance on when to use alternative methods.

---

## Pearson Correlation Assumptions

### 1. **Linear Relationship**
- Variables should be linearly related (straight-line pattern)
- **Test**: Compare linear vs. quadratic model fit (R² values)
- **Violation**: Curved relationships reduce Pearson correlation strength

### 2. **Normal Distribution**
- Both variables should be approximately normally distributed
- **Test**: Shapiro-Wilk test, Q-Q plots
- **Violation**: Skewed data can distort correlation estimates

### 3. **Homoscedasticity**
- Constant variance across the range (equal spread). Basically, whether the spread of the data points around the regression line is consistent
- **Test**: Residuals vs. fitted values plot, Breusch-Pagan test
- **Violation**: Unequal variance can affect reliability

### 4. **No Extreme Outliers**
- Outliers can severely distort correlation coefficients
- **Tests**: Z-scores (>3), Mahalanobis distance for bivariate outliers
- **Violation**: Single outliers can create or hide relationships

### 5. **Independence**
- Observations should be independent
- **Assessment**: Consider data collection method
- **Violation**: Clustered or repeated measures require special handling

---

## What This Analysis Does

### How We Test Each Assumption

#### **Linearity Test**
- **Method**: Fit both linear (degree 1) and quadratic (degree 2) polynomial models
- **Comparison**: Calculate R² for each model
- **Decision rule**: If quadratic R² improvement < 0.05, relationship is sufficiently linear. If adding a curved term only explains 5% more variance, the straight-line relationship is adequate for correlation analysis.
- **Why this works**: Non-linear relationships will show substantial improvement with quadratic terms. Curved relationships are better captured by quadratic models, so a large R² jump (>0.05) indicates the linear correlation is missing important non-linear patterns.

- **Dumbing it down**: We fit a straight line and a curved line to the data and compare how well each explains the relationship. If the curved line only explains 5% more variance than the straight line, we conclude the relationship is linear enough for Pearson correlation..

#### **Normality Test**
- **Method**: Shapiro-Wilk test for each variable
- **Null hypothesis**: Data comes from normal distribution
- **Decision rule**: p > 0.05 suggests normality
- **Sample size limit**: Uses sample of 5000 if dataset is larger
- **Visual check**: Q-Q plots show departures from straight line

- **Dumbing it down**: We run a Shapiro-Wilk statistical test on each variable to check if it follows a normal (bell-shaped) distribution. If the p-value is greater than 0.05, we assume the variable is normal enough for Pearson correlation.

#### **Homoscedasticity Test**
- **Method**: Simplified Breusch-Pagan test
- **Process**: Calculate residuals from linear regression, then correlate |residuals| with fitted values
- **Logic**: If variance is constant, residuals shouldn't correlate with predicted values
- **Decision rule**: p > 0.05 suggests constant variance

- **Dumbing it down**: We calculate residuals (errors) from a linear regression, then check if the absolute residuals correlate with the predicted values. If there's no significant correlation (p > 0.05), it means the variance is constant across all values.

#### **Outlier Detection**
- **Univariate outliers**: Z-scores > 3 (more than 3 standard deviations from mean)
- **Bivariate outliers**: Mahalanobis distance with chi-square threshold
- **Mahalanobis distance**: Measures how far a point is from the center of data in a multi-dimensional space
- **Threshold**: χ² critical value at 99.9% confidence for 2 variables
- **Warning level**: >5% outliers suggests attention needed

- **Dumbing it down**: For univariate outliers, we identify points more than 3 standard deviations from the mean using z-scores. For bivariate outliers, we calculate Mahalanobis distance (which accounts for the correlation between variables) and flag points that exceed the chi-square threshold at 99.9% confidence. If more than 5% of data points are flagged as outliers by either method, we consider this problematic for correlation analysis.

### Visual Diagnostics
- **Scatter plots**: Linear vs. curved relationships
- **Q-Q plots**: Departure from normality
- **Residual plots**: Homoscedasticity violations
- **Outlier plots**: Mahalanobis distance detection

### Recommendations
- **All assumptions met**: Use Pearson correlation confidently
- **Some violations**: Compare Pearson vs. Spearman results
- **Major violations**: Prefer Spearman rank correlation

---

## Spearman's Rank Correlation

### When to Use Spearman
- **Non-normal data**: Robust to distribution shape
- **Non-linear monotonic relationships**: Captures consistent increasing/decreasing trends
- **Outlier presence**: Less sensitive to extreme values
- **Ordinal data**: Works with ranked data

### Interpretation
- **ρ (rho)**: Spearman correlation coefficient
- **Values**: Same range as Pearson (-1 to +1)
- **Meaning**: Strength of monotonic (consistently increasing/decreasing) relationship

### Comparison Strategy
This analysis compares Pearson vs. Spearman correlations to reveal:
- **High agreement**: Linear relationships with normal data
- **Moderate disagreement**: Some non-linearity or outliers present
- **Major disagreement**: Serious assumption violations

When assumptions are violated, **Spearman provides more reliable estimates** of the true relationship strength.

In [ ]:
def check_correlation_assumptions(data, var1, var2, create_plots=True):
    """
    Assumption checking for Pearson correlation between two variables.
    """
    print(f"\n" + "=" * 60)
    print(f"ASSUMPTION CHECKING: {var1} vs {var2}")
    print("=" * 60)

    # Get clean data for the pair
    clean_data = data[[var1, var2]].dropna()
    x = clean_data[var1]
    y = clean_data[var2]
    n = len(clean_data)

    print(f"Sample size: {n} complete pairs")

    # 1. Linear relationship check
    print(f"\n1. LINEAR RELATIONSHIP")
    print("-" * 25)

    # Calculate Pearson correlation
    r_pearson, p_pearson = pearsonr(x, y)

    # Fit polynomial models to test for non-linearity
    #Note: Remember that R-squared indicates how well the model fits the data
    # Linear model (degree 1)
    linear_coef = np.polyfit(x, y, 1)
    linear_pred = np.polyval(linear_coef, x)
    linear_r2 = 1 - (np.sum((y - linear_pred) ** 2) / np.sum((y - np.mean(y)) ** 2))

    # Quadratic model (degree 2)
    quad_coef = np.polyfit(x, y, 2)
    quad_pred = np.polyval(quad_coef, x)
    quad_r2 = 1 - (np.sum((y - quad_pred) ** 2) / np.sum((y - np.mean(y)) ** 2))

    # Test for non-linearity
    r2_improvement = quad_r2 - linear_r2
    linearity_ok = r2_improvement < 0.05  # Arbitrary threshold

    print(f"Linear R² = {linear_r2:.4f}")
    print(f"Quadratic R² = {quad_r2:.4f}")
    print(f"R² improvement = {r2_improvement:.4f}")
    print(f"Linearity assumption: {'SATISFIED' if linearity_ok else 'QUESTIONABLE'}")

    # 2. Normality check
    print(f"\n2. NORMALITY")
    print("-" * 15)

    # Shapiro-Wilk tests (use sample if too large)
    x_sample = x.sample(min(5000, len(x)), random_state=42) if len(x) > 5000 else x
    y_sample = y.sample(min(5000, len(y)), random_state=42) if len(y) > 5000 else y

    shapiro_x = stats.shapiro(x_sample)
    shapiro_y = stats.shapiro(y_sample)

    normality_x = shapiro_x.pvalue > 0.05
    normality_y = shapiro_y.pvalue > 0.05

    print(f"{var1}: W = {shapiro_x.statistic:.4f}, p = {shapiro_x.pvalue:.4f} ({'Normal' if normality_x else 'Non-normal'})")
    print(f"{var2}: W = {shapiro_y.statistic:.4f}, p = {shapiro_y.pvalue:.4f} ({'Normal' if normality_y else 'Non-normal'})")

    # 3. Homoscedasticity check
    print(f"\n3. HOMOSCEDASTICITY")
    print("-" * 20)

    # Calculate residuals from linear regression
    residuals = y - linear_pred

    # Breusch-Pagan test for heteroscedasticity
    # Simple version: correlation between |residuals| and fitted values
    abs_residuals = np.abs(residuals)
    fitted_values = linear_pred

    hetero_corr, hetero_p = pearsonr(fitted_values, abs_residuals)
    homoscedasticity_ok = hetero_p > 0.05

    print(f"Breusch-Pagan-like test:")
    print(f"Correlation |residuals| vs fitted: r = {hetero_corr:.4f}, p = {hetero_p:.4f}")
    print(f"Homoscedasticity assumption: {'SATISFIED' if homoscedasticity_ok else 'VIOLATED'}")

    # 4. Outlier detection
    print(f"\n4. OUTLIER DETECTION")
    print("-" * 20)

    # Z-score method for univariate outliers
    #Note: Remember that Z-score indicates how many standard deviations a data point is from the mean of a distribution
    z_scores_x = np.abs(stats.zscore(x))
    z_scores_y = np.abs(stats.zscore(y))

    outliers_x = np.sum(z_scores_x > 3)
    outliers_y = np.sum(z_scores_y > 3)

    # Bivariate outliers using Mahalanobis distance
    data_array = np.column_stack([x, y])    # Combine x and y into a 2D array
    mean_vec = np.mean(data_array, axis=0)   # Calculate mean of x and mean of y
    cov_matrix = np.cov(data_array.T)    # Calculate covariance matrix

    try:
        inv_cov = np.linalg.inv(cov_matrix)   # Invert the covariance matrix
        mahal_dist = []
        for i in range(len(data_array)):
            diff = data_array[i] - mean_vec   # How far this point is from the center
            mahal_dist.append(np.sqrt(diff.T @ inv_cov @ diff))  # Mahalanobis distance formula

        mahal_dist = np.array(mahal_dist)
        # Chi-square critical value for 2 variables at 99.9% confidence
        chi2_critical = stats.chi2.ppf(0.999, df=2)    # Get chi-square threshold for 99.9% confidence
        bivariate_outliers = np.sum(mahal_dist**2 > chi2_critical)  # Count outliers

    except np.linalg.LinAlgError:
        bivariate_outliers = 0
        mahal_dist = np.zeros(len(data_array))

    total_outliers = max(outliers_x, outliers_y, bivariate_outliers)
    outliers_ok = total_outliers < (0.05 * n)  # Less than 5% outliers

    print(f"Univariate outliers (|z| > 3):")
    print(f"  {var1}: {outliers_x} ({outliers_x/n*100:.1f}%)")
    print(f"  {var2}: {outliers_y} ({outliers_y/n*100:.1f}%)")
    print(f"Bivariate outliers (Mahalanobis): {bivariate_outliers} ({bivariate_outliers/n*100:.1f}%)")
    print(f"Outlier assumption: {'SATISFIED' if outliers_ok else 'ATTENTION NEEDED'}")

    # Create diagnostic plots if requested
    if create_plots:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle(f'Correlation Assumptions: {var1} vs {var2}', fontsize=16)

        # Plot 1: Scatter plot with linear and quadratic fits
        axes[0, 0].scatter(x, y, alpha=0.6, s=30)

        # Linear fit
        x_line = np.linspace(x.min(), x.max(), 100)
        y_linear = np.polyval(linear_coef, x_line)
        y_quad = np.polyval(quad_coef, x_line)

        axes[0, 0].plot(x_line, y_linear, 'r-', linewidth=2, label=f'Linear (R² = {linear_r2:.3f})')
        axes[0, 0].plot(x_line, y_quad, 'g--', linewidth=2, label=f'Quadratic (R² = {quad_r2:.3f})')

        axes[0, 0].set_xlabel(var1)
        axes[0, 0].set_ylabel(var2)
        axes[0, 0].set_title('Linearity Check')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

        # Plot 2: Residuals vs Fitted (homoscedasticity)
        axes[0, 1].scatter(fitted_values, residuals, alpha=0.6, s=30)
        axes[0, 1].axhline(y=0, color='red', linestyle='--')
        axes[0, 1].set_xlabel('Fitted Values')
        axes[0, 1].set_ylabel('Residuals')
        axes[0, 1].set_title('Residuals vs Fitted (Homoscedasticity)')
        axes[0, 1].grid(True, alpha=0.3)

        # Plot 3: Q-Q plot for X variable
        stats.probplot(x, dist="norm", plot=axes[0, 2])
        axes[0, 2].set_title(f'Q-Q Plot: {var1}')
        axes[0, 2].grid(True, alpha=0.3)

        # Plot 4: Q-Q plot for Y variable
        stats.probplot(y, dist="norm", plot=axes[1, 0])
        axes[1, 0].set_title(f'Q-Q Plot: {var2}')
        axes[1, 0].grid(True, alpha=0.3)

        # Plot 5: Outlier detection (Mahalanobis distance)
        if len(mahal_dist) > 0:
            axes[1, 1].scatter(range(len(mahal_dist)), mahal_dist**2, alpha=0.6, s=30)
            axes[1, 1].axhline(y=chi2_critical, color='red', linestyle='--',
                              label=f'Critical value = {chi2_critical:.1f}')
            axes[1, 1].set_xlabel('Observation Index')
            axes[1, 1].set_ylabel('Squared Mahalanobis Distance')
            axes[1, 1].set_title('Bivariate Outlier Detection')
            axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)

        # Plot 6: Scale-Location plot
        sqrt_abs_residuals = np.sqrt(np.abs(residuals))
        axes[1, 2].scatter(fitted_values, sqrt_abs_residuals, alpha=0.6, s=30)
        axes[1, 2].set_xlabel('Fitted Values')
        axes[1, 2].set_ylabel('√|Residuals|')
        axes[1, 2].set_title('Scale-Location Plot')
        axes[1, 2].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    # Summary
    assumptions_met = sum([linearity_ok, normality_x, normality_y, homoscedasticity_ok, outliers_ok])
    total_assumptions = 5

    print(f"\n" + "=" * 50)
    print("ASSUMPTION CHECK SUMMARY")
    print("=" * 50)
    print(f"Assumptions satisfied: {assumptions_met}/{total_assumptions}")

    if assumptions_met >= 4:
        recommendation = "Pearson correlation is appropriate"
    elif assumptions_met >= 2:
        recommendation = "Consider both Pearson and Spearman correlations"
    else:
        recommendation = "Spearman correlation recommended"

    print(f"Recommendation: {recommendation}")

    return {
        'linearity': linearity_ok,
        'normality_x': normality_x,
        'normality_y': normality_y,
        'homoscedasticity': homoscedasticity_ok,
        'outliers_ok': outliers_ok,
        'assumptions_met': assumptions_met,
        'recommendation': recommendation,
        'pearson_r': r_pearson,
        'sample_size': n,
        'outlier_counts': {
            'univariate_x': outliers_x,
            'univariate_y': outliers_y,
            'bivariate': bivariate_outliers
        }
    }


def perform_spearman_correlation(data, continuous_vars):
    """
    Perform Spearman's rank correlation analysis as a non-parametric alternative.
    """
    print(f"\n" + "=" * 70)
    print("SPEARMAN'S RANK CORRELATION ANALYSIS")
    print("=" * 70)

    print("Spearman's rank correlation (ρ) measures monotonic relationships")
    print("It's robust to outliers and doesn't assume normality or linearity")
    print()

    # Calculate Spearman correlation matrix - using pandas .corr function - all pairs results simultaneously
    # Note: The .corr(method='spearman' method automatically converts data to ranks
    spearman_matrix = data[continuous_vars].corr(method='spearman')

    print("Spearman Correlation Matrix:")
    print("-" * 30)
    print(spearman_matrix.round(3))

    # Compare with Pearson correlations
    pearson_matrix = data[continuous_vars].corr(method='pearson')

# Just so we can print p-values, let us do this individually, for each pair of variables at a time
# using the scipy.stats functions, which returns the p-values
    print(f"\nSpearman vs Pearson Correlations:")
    print("-" * 35)

    spearman_results = []

    for i in range(len(continuous_vars)):
        for j in range(i+1, len(continuous_vars)):
            var1 = continuous_vars[i]
            var2 = continuous_vars[j]

            # Get aligned data first, then calculate correlations
            clean_data = data[[var1, var2]].dropna()
            x = clean_data[var1]
            y = clean_data[var2]

            # Spearman correlation (using function from scipy.stats)
            rho, p_spearman = spearmanr(x, y)

            # Pearson correlation for comparison (using function from scipy.stats)
            r_pearson, p_pearson = pearsonr(x, y)

            # Calculate difference
            diff = abs(rho - r_pearson)

            spearman_results.append({
                'var1': var1,
                'var2': var2,
                'spearman_rho': rho,
                'spearman_p': p_spearman,
                'pearson_r': r_pearson,
                'pearson_p': p_pearson,
                'difference': diff,
                'sample_size': len(clean_data)
            })

            # Interpret difference
            if diff < 0.1:
                agreement = "High agreement"
            elif diff < 0.2:
                agreement = "Moderate agreement"
            else:
                agreement = "Poor agreement"

            print(f"{var1} ↔ {var2}:")
            print(f"  Spearman ρ = {rho:>7.3f} (p = {p_spearman:>6.4f})")
            print(f"  Pearson  r = {r_pearson:>7.3f} (p = {p_pearson:>6.4f})")
            print(f"  Difference = {diff:>7.3f} ({agreement})")
            print()

    # Create comparison visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Spearman vs Pearson Correlation Comparison', fontsize=16)

    # Plot 1: Spearman correlation heatmap
    #mask = np.triu(np.ones_like(spearman_matrix, dtype=bool))
    sns.heatmap(spearman_matrix,
                #mask=mask,
                annot=True, cmap='RdBu_r', center=0,
                square=True, fmt='.3f', ax=axes[0, 0],
                cbar_kws={'label': 'Spearman ρ'})
    axes[0, 0].set_title('Spearman Correlation Matrix')

    # Plot 2: Pearson correlation heatmap
    sns.heatmap(pearson_matrix,
                #mask=mask,
                annot=True, cmap='RdBu_r', center=0,
                square=True, fmt='.3f', ax=axes[0, 1],
                cbar_kws={'label': 'Pearson r'})
    axes[0, 1].set_title('Pearson Correlation Matrix')

    # Plot 3: Spearman vs Pearson scatter plot
    spearman_values = []
    pearson_values = []

    for result in spearman_results:
        spearman_values.append(result['spearman_rho'])
        pearson_values.append(result['pearson_r'])

    axes[1, 0].scatter(pearson_values, spearman_values, alpha=0.7, s=60)

    # Add diagonal line (perfect agreement)
    min_val = min(min(spearman_values), min(pearson_values))
    max_val = max(max(spearman_values), max(pearson_values))
    axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--',
                   linewidth=2, label='Perfect agreement')

    axes[1, 0].set_xlabel('Pearson r')
    axes[1, 0].set_ylabel('Spearman ρ')
    axes[1, 0].set_title('Spearman vs Pearson Correlations')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Plot 4: Difference magnitude histogram
    differences = [result['difference'] for result in spearman_results]
    axes[1, 1].hist(differences, bins=10, alpha=0.7, color='skyblue', edgecolor='black')
    axes[1, 1].set_xlabel('|Spearman ρ - Pearson r|')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Distribution of Correlation Differences')
    axes[1, 1].grid(True, alpha=0.3)

    # Add statistics
    mean_diff = np.mean(differences)
    max_diff = np.max(differences)
    axes[1, 1].axvline(mean_diff, color='red', linestyle='--',
                      label=f'Mean difference = {mean_diff:.3f}')
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()

    # Summary statistics
    print(f"Comparison Summary:")
    print("-" * 20)
    print(f"Mean absolute difference: {mean_diff:.3f}")
    print(f"Maximum difference: {max_diff:.3f}")
    print(f"Number of pairs with |diff| > 0.1: {sum(1 for d in differences if d > 0.1)}")
    print(f"Number of pairs with |diff| > 0.2: {sum(1 for d in differences if d > 0.2)}")

    return {
        'spearman_matrix': spearman_matrix,
        'pairwise_results': spearman_results,
        'comparison_stats': {
            'mean_difference': mean_diff,
            'max_difference': max_diff,
            'differences': differences
        }
    }


def analyze_penguin_correlations_comprehensive(data, continuous_vars):
    """
    Comprehensive correlation analysis of penguin data with both methods.
    """
    print(f"\n" + "=" * 70)
    print("COMPREHENSIVE PENGUIN CORRELATION ANALYSIS")
    print("=" * 70)

    # Check assumptions for key variable pairs
    key_pairs = [
        ('flipper_length_mm', 'body_mass_g'),
        ('bill_length_mm', 'bill_depth_mm'),
        ('bill_length_mm', 'body_mass_g')
    ]

    assumption_results = {}

    for var1, var2 in key_pairs:
        print(f"\n Analyzing {var1} vs {var2}")
        assumption_results[f"{var1}_{var2}"] = check_correlation_assumptions(
            data, var1, var2, create_plots=True
        )

    # Summary of assumption checks
    print(f"\n" + "=" * 50)
    print("ASSUMPTION CHECK SUMMARY FOR ALL PAIRS")
    print("=" * 50)

    for pair_name, results in assumption_results.items():
        # FIXED: More robust string splitting
        parts = pair_name.split('_')
        # Reconstruct variable names properly
        if len(parts) >= 4:  # e.g., "bill_length_mm_body_mass_g"
            var1 = '_'.join(parts[:-3])  # Everything except last 3 parts
            var2 = '_'.join(parts[-3:])  # Last 3 parts
        else:
            var1, var2 = pair_name.split('_', 1)

        var1_clean = var1.replace('_', ' ')
        var2_clean = var2.replace('_', ' ')

        print(f"\n{var1_clean} ↔ {var2_clean}:")
        print(f"  Assumptions met: {results['assumptions_met']}/5")
        print(f"  Recommendation: {results['recommendation']}")
        print(f"  Pearson r = {results['pearson_r']:.3f}")

    return assumption_results


# Execute the analysis
print("\n EXECUTING ASSUMPTION CHECKING AND SPEARMAN ANALYSIS")
print("=" * 60)

# Step 1: Comprehensive assumption checking
print("\n Step 1: Checking correlation assumptions for key variable pairs")
penguin_assumptions = analyze_penguin_correlations_comprehensive(penguins_clean, continuous_vars)

# Step 2: Spearman correlation analysis
print("\n Step 2: Spearman correlation analysis")
spearman_results = perform_spearman_correlation(penguins_clean, continuous_vars)